# Phase 2: Explore the ciphers

This notebook compares ciphertext structure only. It does not attempt decryption.

In [ ]:
from collections import Counter
from pathlib import Path
import json
import math
import matplotlib.pyplot as plt

root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
paths = sorted((root / 'data' / 'raw').glob('z*.json'))
ciphers = {}
for path in paths:
    record = json.loads(path.read_text())
    ciphers[record['cipher']] = [symbol for row in record['rows'] for symbol in row]

assert {name: len(symbols) for name, symbols in ciphers.items()} == {'Z13': 13, 'Z32': 32, 'Z340': 340, 'Z408': 408}
list(ciphers)

In [ ]:
def repeated_ngrams(symbols, n):
    counts = Counter(tuple(symbols[i:i + n]) for i in range(len(symbols) - n + 1))
    return {gram: count for gram, count in counts.items() if count > 1}

def metrics(symbols):
    counts = Counter(symbols)
    size = len(symbols)
    probabilities = [count / size for count in counts.values()]
    return {
        'length': size,
        'unique_symbols': len(counts),
        'entropy_bits': -sum(p * math.log2(p) for p in probabilities),
        'index_of_coincidence': sum(count * (count - 1) for count in counts.values()) / (size * (size - 1)),
        'repeated_bigrams': len(repeated_ngrams(symbols, 2)),
        'repeated_trigrams': len(repeated_ngrams(symbols, 3)),
    }

results = {name: metrics(symbols) for name, symbols in ciphers.items()}
for name, values in results.items():
    print(name, {key: round(value, 4) if isinstance(value, float) else value for key, value in values.items()})

In [ ]:
for name, symbols in ciphers.items():
    print(f'{name} most common symbols:', Counter(symbols).most_common(10))
    for n in (2, 3):
        repeats = sorted(repeated_ngrams(symbols, n).items(), key=lambda item: (-item[1], item[0]))
        print(f'{name} repeated {n}-grams:', repeats[:10] or 'none')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for axis, (name, symbols) in zip(axes.flat, ciphers.items()):
    frequencies = Counter(symbols).most_common()
    axis.bar(range(len(frequencies)), [count for _, count in frequencies])
    axis.set(title=f'{name} symbol frequencies', xlabel='Symbols, most to least frequent', ylabel='Count')
    axis.set_xticks([])
fig.tight_layout()
plt.show()

## Observations

In [ ]:
solved = ('Z408', 'Z340')
unsolved = ('Z13', 'Z32')
print(f"The solved ciphers are much longer ({results['Z340']['length']} and {results['Z408']['length']} symbols) than the unsolved ciphers ({results['Z13']['length']} and {results['Z32']['length']}).")
for name in solved + unsolved:
    values = results[name]
    print(f"{name}: {values['unique_symbols']} unique symbols, entropy {values['entropy_bits']:.3f} bits, IC {values['index_of_coincidence']:.4f}, {values['repeated_bigrams']} repeated bigrams, and {values['repeated_trigrams']} repeated trigrams.")
print('Because Z13 and Z32 are short, their frequency, entropy, coincidence, and repetition estimates are less stable; these measurements alone do not support a decryption claim.')